# Jean Paul Collazo Model Testing
## Random Forest & Neural Network
Dataset  
https://archive.ics.uci.edu/dataset/2/adult

Environment based on Anaconda (Python >= 3.13) and scikit-learn.

This notebook evaluates Random Forest and Multi-Layer Perceptron (MLP) classifiers using the same preprocessing steps and train/test split for a fair comparison.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier


In [ ]:
all_columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain',
    'capital-loss', 'hours-per-week', 'native-country', 'income']
categorical_features = ['workclass', 'education', 'marital-status', 'occupation',
    'relationship', 'race', 'sex', 'native-country']
numerical_features = ['age', 'fnlwgt', 'education-num', 'capital-gain',
    'capital-loss', 'hours-per-week']
target_feature = 'income'
random_state_value = 48

df = pd.read_csv('adult.data.csv', header=None, names=all_columns)
df.head()


In [ ]:
# Replace question marks with missing values and remove extra spaces.
for column in categorical_features + [target_feature]:
    df[column] = df[column].str.strip()

df.replace('?', np.nan, inplace=True)

X = df[categorical_features + numerical_features]
y = df[target_feature].apply(lambda x: 1 if x == '>50K' else 0)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=random_state_value,
    stratify=y
)

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)


In [ ]:
# Random Forest model using the shared preprocessing pipeline
random_forest = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight='balanced',
    random_state=random_state_value,
    n_jobs=-1
)

rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', random_forest)
])

rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)
rf_probabilities = rf_model.predict_proba(X_test)[:, 1]


In [ ]:
rf_confusion = confusion_matrix(y_test, rf_predictions)
print('Random Forest Confusion Matrix:')
print(rf_confusion)


In [ ]:
print()
print('Random Forest Classification Report:')
print(classification_report(y_test, rf_predictions))
print()
print('Random Forest Predicted Probabilities:')
print(rf_probabilities)
print('Random Forest Accuracy:', accuracy_score(y_test, rf_predictions))
print('Random Forest Precision:', precision_score(y_test, rf_predictions))
print('Random Forest Recall:', recall_score(y_test, rf_predictions))
print('Random Forest F1 Score:', f1_score(y_test, rf_predictions))
print('Random Forest ROC-AUC:', roc_auc_score(y_test, rf_probabilities))


In [ ]:
# Neural Network model using the same preprocessing pipeline
neural_network = MLPClassifier(
    hidden_layer_sizes=(100,),
    activation='relu',
    solver='adam',
    alpha=0.0001,
    learning_rate_init=0.001,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=15,
    random_state=random_state_value
)

nn_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', neural_network)
])

nn_model.fit(X_train, y_train)
nn_predictions = nn_model.predict(X_test)
nn_probabilities = nn_model.predict_proba(X_test)[:, 1]


In [ ]:
nn_confusion = confusion_matrix(y_test, nn_predictions)
print('Neural Network Confusion Matrix:')
print(nn_confusion)


In [ ]:
print()
print('Neural Network Classification Report:')
print(classification_report(y_test, nn_predictions))
print()
print('Neural Network Predicted Probabilities:')
print(nn_probabilities)
print('Neural Network Accuracy:', accuracy_score(y_test, nn_predictions))
print('Neural Network Precision:', precision_score(y_test, nn_predictions))
print('Neural Network Recall:', recall_score(y_test, nn_predictions))
print('Neural Network F1 Score:', f1_score(y_test, nn_predictions))
print('Neural Network ROC-AUC:', roc_auc_score(y_test, nn_probabilities))


In [ ]:
# Compare model performance in one table
model_comparison = pd.DataFrame({
    'Model': ['Random Forest', 'Neural Network'],
    'Accuracy': [
        accuracy_score(y_test, rf_predictions),
        accuracy_score(y_test, nn_predictions)
    ],
    'Precision': [
        precision_score(y_test, rf_predictions),
        precision_score(y_test, nn_predictions)
    ],
    'Recall': [
        recall_score(y_test, rf_predictions),
        recall_score(y_test, nn_predictions)
    ],
    'F1 Score': [
        f1_score(y_test, rf_predictions),
        f1_score(y_test, nn_predictions)
    ],
    'ROC-AUC': [
        roc_auc_score(y_test, rf_probabilities),
        roc_auc_score(y_test, nn_probabilities)
    ]
})

model_comparison.round(4)


In [ ]:
# Optional confusion-matrix visualizations
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(rf_confusion, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Random Forest Confusion Matrix')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

sns.heatmap(nn_confusion, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title('Neural Network Confusion Matrix')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')

plt.tight_layout()
plt.show()
